# Stage 4, chunk 6: chat SFT of the adapted model

LoRA fine-tuning of the merged Stage 3 model on the filtered teacher conversations, with the loss on the assistant's reply only, so the model answers again with thinking on and replies in Urdu. All logic lives in `zubaan.tune.sft`; decisions in `docs/decisions.md` (2026-11-26).

**Two modes, in this order** (first code cell):

1. `smoke`: 20 steps, local checkpoints deleted, resumed from the Hub to step 30. Reports speed, peak memory, the held-out loss and the main run's projected hours. **Send back `smoke_report.json`.**
2. `main`: the full schedule (2 epochs), then merge, save `tuned/model`, and a first look at a few replies. Checkpoints go to the private Hub repo every 25 steps; if the session ends, run it again with the same settings and it resumes.

Settings before running:

- Accelerator: **GPU T4 x2** (one GPU is used). Internet: **on**.
- Inputs (**Add Input**): the output of `07_adapt_main_qwen3` (the Stage 3 model), and a **dataset** with `conversations.jsonl` (and `curate_report.json`) from `zubaan tune filter`.
- Secret: **HF_TOKEN** (Add-ons > Secrets).
- Run with **Save Version > Save & Run All (Commit)**.

In [ ]:
# Run MODE = "smoke" first (about 30 to 45 minutes), send back smoke_report.json, then run MODE = "main".
MODE = "smoke"
# Private Hub repo for Stage 4 checkpoints (a different one from Stage 3's). Smoke runs use HUB_REPO + "-smoke".
HUB_REPO = "hashiraziz21/zubaan-qwen3-tune-ckpt"            # e.g. "your-name/zubaan-qwen3-tune-ckpt"
GRAD_CHECKPOINT = True   # rows are 4,096 tokens (Stage 3 used 2 x 1,024); the smoke run reports peak memory
EPOCHS = 2
LR_LORA = 1e-4
TRAIN_NEW_ROWS = False   # the plan: start with the Stage 2 rows frozen
# Do not change the lines above between sessions of the same run: a resumed run refuses a changed schedule.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # before torch starts
assert MODE in ("smoke", "main"), "MODE must be smoke or main"
assert HUB_REPO, "set HUB_REPO: without it a crash or the 12-hour limit loses the run"

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "c5c1309"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,train] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
# Kaggle ships torchao 0.10, which peft refuses ("only versions above 0.16 are supported") when it
# builds LoRA layers. We never use torchao, so remove it; peft then skips its torchao support.
!pip uninstall -y -q torchao
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.adapt.pipeline  # noqa: F401

print("OK: code version verified")

In [ ]:
import os
import subprocess

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: a GPU with about 16 GB. Training uses one GPU (fp16: a T4 and a P100 have no BF16).
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator in the notebook settings"
print(r.stdout)
gpu_name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(mem.split()[0]) >= 14000, f"{gpu_name} has {mem}; need about 16 GB"
if float(cap) < 7.0:
    print("WARNING: compute capability", cap, "has no fp16 tensor cores; this will be slow. A T4 is better.")

import peft
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("OK: GPU", gpu_name, "compute capability", cap, mem)

In [ ]:
# The Hub token is a Kaggle secret named HF_TOKEN (Add-ons > Secrets). It is never written in code.
if HUB_REPO:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hub checkpoints go to the PRIVATE repo", HUB_REPO, "(smoke runs use", HUB_REPO + "-smoke)")
else:
    print("No Hub repo: checkpoints stay in this session. A crash or the 12-hour limit loses the run.")

## 1. Inputs: the Stage 3 model and the conversations

In [ ]:
import json
from pathlib import Path

# Two inputs (Add Input in the sidebar):
#  1. the output of notebook 07 (the merged Stage 3 model, adapted/model), and
#  2. a Kaggle dataset holding conversations.jsonl (from zubaan tune filter on the laptop).
INPUT_ROOT = Path("/kaggle/input")
stage3 = []
for m in sorted(INPUT_ROOT.rglob("zubaan_extension.json")):
    meta = json.loads(m.read_text(encoding="utf-8"))
    if meta.get("trained") and "stage4" not in meta:
        stage3.append(m.parent)
assert len(stage3) == 1, f"need exactly one Stage 3 model (output of notebook 07), found {[str(s) for s in stage3]}"
MODEL_DIR = stage3[0]
assert (MODEL_DIR / "model.safetensors").exists() or list(MODEL_DIR.glob("model-*.safetensors")), "no weights"
data_files = sorted(INPUT_ROOT.rglob("conversations.jsonl"))
assert len(data_files) == 1, f"need exactly one conversations.jsonl (upload it as a dataset), found {data_files}"
TRAIN_FILE = data_files[0]

from transformers import AutoTokenizer

assert AutoTokenizer.from_pretrained(MODEL_DIR).chat_template, "the Stage 3 folder has no chat template"
report = TRAIN_FILE.parent / "curate_report.json"
print("model:", MODEL_DIR)
print("conversations:", TRAIN_FILE, sum(1 for _ in TRAIN_FILE.open(encoding="utf-8")), "lines")
if report.exists():
    print("filter report:", json.loads(report.read_text(encoding="utf-8"))["slots"])

## 2. Configuration and Hub check

In [ ]:
import yaml

WORK = Path("/kaggle/working")
OUT = WORK / ("tuned" if MODE == "main" else "tune_smoke")
# Smoke runs get their own Hub repo, so the main run can never resume from a smoke checkpoint.
HUB = HUB_REPO if MODE == "main" else HUB_REPO + "-smoke"
tune = {
    "model_path": str(MODEL_DIR),
    "train_file": str(TRAIN_FILE),
    "max_len": 4096,
    "val_share": 0.02,
    "output_dir": str(OUT),
    "epochs": EPOCHS,
    "micro_batch_size": 1,
    "grad_accum": 16,
    "lr_lora": LR_LORA,
    "train_new_rows": TRAIN_NEW_ROWS,
    "warmup_steps": 20,
    "lora_rank": 64,
    "lora_alpha": 128,
    "precision": "fp16",
    "gradient_checkpointing": GRAD_CHECKPOINT,
    "device": "cuda",
    "seed": 0,
    "log_every": 5,
    "eval_every": 50,
    "checkpoint_every": 25,
    "keep_checkpoints": 2,
    "resume": True,
    "hub_repo": HUB,
    "hub_private": True,
}
Path("tune.yaml").write_text(yaml.safe_dump({"tune": tune}, sort_keys=False))
from zubaan.config import load_config

cfg = load_config("tune.yaml").tune
print(cfg.model_dump_json(indent=1))

In [ ]:
# Before training: the repo exists, is private, accepts a write, and says where this run starts.
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HUB, private=True, exist_ok=True)
assert api.repo_info(HUB).private, f"{HUB} is public: make it private on the Hub first"
api.upload_file(path_or_fileobj=b"ok", path_in_repo="_preflight/ok.txt", repo_id=HUB)
api.delete_folder("_preflight", repo_id=HUB)
files = set(api.list_repo_files(HUB))
if MODE == "smoke" and "last/meta.json" in files:
    # An earlier smoke run's checkpoint; the smoke test must start at step 0 to test the resume.
    api.delete_folder("last", repo_id=HUB)
    print("removed an earlier smoke checkpoint from", HUB)
elif "last/meta.json" in files:
    from huggingface_hub import hf_hub_download

    step = json.loads(Path(hf_hub_download(HUB, "last/meta.json")).read_text())["step"]
    print(f"OK: a checkpoint at step {step} exists in {HUB}: this run RESUMES from it.")
else:
    print(f"OK: {HUB} is private and writable. No checkpoint yet: this run starts from step 0.")

## 3. Smoke run (MODE = smoke)

In [ ]:
# Smoke: 20 steps, delete the local checkpoints, resume from the Hub to step 30. Then speed, memory,
# the held-out loss, and the main run's projected hours.
import gc
import shutil
import statistics

import torch

from zubaan.tune.sft import run_tune


def free_gpu():
    # The model of a finished run_tune call is garbage; release its GPU memory before the next one.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if MODE == "smoke":
    run_tune(cfg, log=print, stop_at_step=20)
    free_gpu()
    shutil.rmtree(cfg.output_dir / "checkpoints")
    second = run_tune(cfg, log=print, stop_at_step=30)
    assert second["resumed_from_step"] == 20, "the run did not resume from the Hub checkpoint"
    log_rows = [json.loads(x) for x in (cfg.output_dir / "train_log.jsonl").read_text().splitlines()]
    steps = [r for r in log_rows if "loss" in r]
    speed = statistics.median(r["tokens_per_s"] for r in steps[1:])
    train = second["data"]["train"]
    tokens_per_step = train["tokens"] / train["rows"] * cfg.micro_batch_size * cfg.grad_accum
    smoke = {
        "gpu": gpu_name,
        "data": second["data"],
        "packing_max_logit_diff": second["packing_max_logit_diff"],
        "losses": [(r["step"], round(r["loss"], 4)) for r in steps],
        "evals": second["evals"],
        "median_tokens_per_s": round(speed),
        "peak_gpu_mem_gb": round(max(r.get("gpu_mem_gb", 0) for r in steps), 2),
        "total_steps": second["total_steps"],
        "projected_main_hours": round(second["total_steps"] * tokens_per_step / speed / 3600, 2),
        "settings": {"grad_checkpoint": GRAD_CHECKPOINT, "epochs": EPOCHS, "lr_lora": LR_LORA, "train_new_rows": TRAIN_NEW_ROWS},
    }
    (WORK / "smoke_report.json").write_text(json.dumps(smoke, indent=1, ensure_ascii=False))
    print(json.dumps({k: v for k, v in smoke.items() if k != "data"}, indent=1))
    print("Send back smoke_report.json (Output tab).")

## 4. Main run, merge and a first look (MODE = main)

In [ ]:
# Main: the full schedule. If the session ends first, run the notebook again with the same settings.
if MODE == "main":
    summary = run_tune(cfg, log=print)
    print({k: summary[k] for k in ("resumed_from_step", "final_step", "total_steps", "finished")})
    assert summary["finished"], "the schedule did not finish: run this notebook again with the same settings to resume"
    for e in summary["evals"]:
        print(e["step"], e["eval"])

In [ ]:
# Fold LoRA into the Stage 3 weights (float32), check the logits did not change, save tuned/model.
if MODE == "main":
    from zubaan.tune.sft import merge_tuned

    free_gpu()
    merged = merge_tuned(cfg, log=print)
    shutil.rmtree(cfg.output_dir / "checkpoints", ignore_errors=True)  # they are on the Hub

In [ ]:
# A first look (not the evaluation): greedy replies to 2 Urdu and 2 English chat-check prompts,
# thinking on and off. The Stage 3 model answered none of these with thinking on.
if MODE == "main":
    from zubaan.tune.sft import quick_probe

    free_gpu()
    probe = quick_probe(Path(merged["model_dir"]), log=print)
    (WORK / "probe.json").write_text(json.dumps(probe, indent=1, ensure_ascii=False))
    for p in probe:
        print("\n===", p["id"], "thinking" if p["thinking"] else "no thinking", "===")
        print(p["reply"][:1500])

## 5. Output size

In [ ]:
total = 0
for f in sorted(WORK.rglob("*")):
    if f.is_file() and f.stat().st_size > 5e6:
        total += f.stat().st_size
        print(f"{f.stat().st_size / 1e9:6.2f} GB  {f.relative_to(WORK)}")
print(f"total of files over 5 MB: {total / 1e9:.1f} GB")
assert total < 19e9, "output is close to Kaggle's 20 GB limit: delete data/raw or old checkpoints"